# YOLO26s 단일 클래스 알약 탐지기 학습

검증된 `data/detector` 패키지를 YOLO 형식의 파생 view로 변환하고, Ultralytics YOLO26s를 학습·검증한다. 기존 TorchVision 탐지기와 비교하기 전의 실험 노트북이므로 기본 설정은 MPS에서 1 epoch 파일럿 실행이다. 파일럿이 정상 완료되면 설정 셀에서 `pilot_mode = False`로 바꾸고 새 `run_name`으로 본 학습을 실행한다.

주요 결과는 `outputs/detector/<run_name>/`의 `weights/best.pt`, `weights/last.pt`, `results.csv`, `resolved_config.json`에 저장된다. 최초 실행에서는 `yolo26s.pt` 가중치를 내려받을 수 있다.

In [ ]:
# ruff: noqa: E402
import sys
from dataclasses import replace
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image
from ultralytics import YOLO

from configs.yolo_detector import load_yolo_detector_config
from data_loading.detector_package import load_detector_package
from detector.yolo_training import train_yolo_detector

## 실행 설정

노트북 값은 YAML 기본값을 덮어쓴다. 파일럿과 본 학습은 반드시 서로 다른 `run_name`을 사용한다.

In [ ]:
pilot_mode = False
run_name = "20261104-yolo26s-det-default-baseline-01"
# tags = [
#     "task:detector",
#     f"stage:{'smoke' if pilot_mode else 'baseline'}",
#     "model:yolo26s",
#     "data:det-default",
#     "change:baseline",
#     "device:mps",
# ]

base_config = load_yolo_detector_config(
    project_root / "configs" / "yolo_detector.yaml"
)
config = replace(
    base_config,
    run_name=run_name,
    output_dir=project_root / "outputs" / "detector",
    data_dir=project_root / "data" / "detector",
    device="mps",
    batch_size=4,
    max_epochs=30,
    close_mosaic_epochs=5,
)
print(f"run_name: {config.run_name}")
print(f"model: {config.model_name} / device: {config.device}")
print(f"epochs: {config.max_epochs} / batch: {config.batch_size}")
print(f"run_dir: {config.run_dir}")

## 입력 검증과 bbox 확인

원본 annotation, split manifest, label map과 data summary의 일치 여부를 먼저 검사한다. validation 첫 이미지에서 bbox 좌표도 육안으로 확인한다. YOLO label view는 학습 함수가 이 검증을 다시 수행한 뒤 갱신한다.

In [ ]:
records_by_split, label_map, data_summary = load_detector_package(config.data_dir)
print(
    f"입력 검증 완료: train={data_summary['num_train_images']}, "
    f"validation={data_summary['num_val_images']}, "
    f"classes={label_map['num_foreground_classes']}"
)
preview_record = records_by_split["validation"][0]
preview_image = Image.open(preview_record["image_path"]).convert("RGB")
figure, axis = plt.subplots(figsize=(10, 7))
axis.imshow(preview_image)
for x, y, width, height in preview_record["bbox_xywh"]:
    axis.add_patch(
        Rectangle((x, y), width, height, fill=False, color="lime", linewidth=2)
    )
axis.set_title(f"GT image_id={preview_record['image_id']}")
axis.axis("off")
figure.tight_layout()

## 학습

같은 실행 폴더가 이미 있으면 덮어쓰지 않고 중단한다. 다시 시작할 때는 `run_name`의 마지막 순번을 올린다.

In [ ]:
training_result = train_yolo_detector(config)
training_result

## 학습 지표

Ultralytics가 저장한 loss와 validation mAP 열을 자동으로 골라 확인한다.

In [ ]:
metrics = pd.read_csv(training_result.metrics_path)
metrics.columns = metrics.columns.str.strip()
loss_columns = [column for column in metrics if "loss" in column]
map_columns = [column for column in metrics if "mAP" in column]
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics.plot(x="epoch", y=loss_columns, ax=axes[0], title="Training losses")
metrics.plot(x="epoch", y=map_columns, ax=axes[1], title="Validation mAP")
figure.tight_layout()
metrics.tail()

## best checkpoint 재평가

mAP 계산에서는 낮은 confidence 예측까지 필요하므로 `score_threshold`를 전달하지 않는다. 해당 값은 아래 시각화와 실제 추론에 사용한다.

In [ ]:
best_model = YOLO(training_result.best_checkpoint_path)
validation_metrics = best_model.val(
    data=str(training_result.dataset_yaml_path),
    imgsz=config.image_size,
    batch=config.batch_size,
    device=config.device,
    workers=config.workers,
    plots=True,
    project=str(training_result.run_dir),
    name="validation",
    exist_ok=True,
    verbose=config.show_progress,
)
validation_metrics.results_dict

## validation 예측 시각화

앞에서 확인한 validation 이미지에 `score_threshold`와 NMS IoU 기준을 적용한다.

In [ ]:
prediction = best_model.predict(
    source=preview_record["image_path"],
    imgsz=config.image_size,
    conf=config.score_threshold,
    iou=config.nms_threshold,
    device=config.device,
    verbose=False,
)[0]
figure, axis = plt.subplots(figsize=(10, 7))
axis.imshow(prediction.plot()[..., ::-1])
axis.set_title(
    f"YOLO26s prediction: {len(prediction.boxes)} boxes "
    f"(conf >= {config.score_threshold})"
)
axis.axis("off")
figure.tight_layout()